In [1]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

c:\llamaindex\ch01\ch01_env\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


In [2]:
documents = SimpleDirectoryReader("data").load_data()

In [3]:
# VectorStoreIndex를 이용하여 인덱스 생성
index = VectorStoreIndex.from_documents(documents)

In [4]:
query_engine = index.as_query_engine()

In [7]:
response=query_engine.query("작가의 청소년 시절은 어떠했는지 한국어로 설명해 줘")

In [8]:
print(response)

작가의 청소년 시절에는 글쓰기와 프로그래밍에 많은 관심을 보였습니다. 9학년 때 IBM 1401 컴퓨터를 사용하여 프로그래밍을 시작했고, 이후에는 마이크로컴퓨터로의 전환으로 프로그래밍에 더욱 열중했습니다. 대학에서는 철학을 전공할 계획이었지만, 철학 수업이 지루해서 인공지능으로 전과하게 되었습니다. 인공지능에 대한 열정은 Heinlein의 소설 'The Moon is a Harsh Mistress'와 PBS 다큐멘터리에서 영향을 받았습니다.


In [15]:
# 라마인덱스: RAG 시스템 구축에 필요한 일련의 과정을 지원
"""
1. 데이터 로딩 (데이터 리더로 데이터로드 -> 정제 및 구조화 -> 문서 객체 형태로 변환)
2. 텍스트 분할 
    2-1. 로드된 문서를 LLM에 최적화된 크기로 나눔
    2-2. 문서 컨텍스트 유지, 검색 효율성 높임. (문장 단위, 토큰 단위, 의미 기반 분할 방식 활용)
        2-2-1. 문서 객체는 청크 단위로 나누어짐. => 노드 객체 생성 (이 때, 노드는 검색, 인덱싱, 질의 처리 과정에서 활용된다.)
3. 인덱싱: 벡터 데이터베이스에 데이터를 저장하기 위한 인덱스 생성. (이 때 생성된 인덱스는 검색(Retrieval) 및 유사도 기반 질의 처리에 사용.)
4. 저장: 외부 벡터 스토어에 연동하여 확장성을 높이고, 데이터 관리 지원.
5. 쿼리: 사용자의 질의를 처리하고 관련 데이터 검색, LLM을 활용하여 자연어 형태로 입력된 쿼리의 의미를 정교하게 해석.
6. 검색: 인덱스를 통해 가장 관련성 높은 결과 검색, 검색된 결과는 추가적인 LLM 기반 후처리를 거쳐, 응답으로 정제된다.
"""

'\n1. 데이터 로딩 (데이터 리더로 데이터로드 -> 정제 및 구조화 -> 문서 객체 형태로 변환)\n2. 텍스트 분할 \n    2-1. 로드된 문서를 LLM에 최적화된 크기로 나눔\n    2-2. 문서 컨텍스트 유지, 검색 효율성 높임. (문장 단위, 토큰 단위, 의미 기반 분할 방식 활용)\n        2-2-1. 문서 객체는 청크 단위로 나누어짐. => 노드 객체 생성 (이 때, 노드는 검색, 인덱싱, 질의 처리 과정에서 활용된다.)\n3. 인덱싱: 벡터 데이터베이스에 데이터를 저장하기 위한 인덱스 생성. (이 때 생성된 인덱스는 검색(Retrieval) 및 유사도 기반 질의 처리에 사용.)\n4. 저장: 외부 벡터 스토어에 연동하여 확장성을 높이고, 데이터 관리 지원.\n5. 쿼리: 사용자의 질의를 처리하고 관련 데이터 검색, LLM을 활용하여 자연어 형태로 입력된 쿼리의 의미를 정교하게 해석.\n6. 검색: 인덱스를 통해 가장 관련성 높은 결과 검색, 검색된 결과는 추가적인 LLM 기반 후처리를 거쳐, 응답으로 정제된다.\n'

In [ ]:

### 예제: 
## - 데이터 로딩: 각 리뷰를 문서 객체로 변환
#  예) "이 영화는 매우 지루했습니다. 줄거리가 너무 느렸어요.
#- 텍스트 분할: 리뷰를 문장 단위로 분할(chunking)하여 각각을 노드 객체로 변환.
#  1. 이 영화는 매우 지루했습니다.
#  2. 줄거리가 너무 느렸어요.
#- 인덱싱: 생성된 노드들을 벡터화하여 벡터 DB에 저장. (이 데이터베이스?는 각 문장의 의미를 기반으로 유사한 내용을 검색할 수 있도록 설계.)
#- 저장: 생성된 벡터 데이터를 영구적으로 저장 및 구성.
#- 질의와 답변 생성:
# 사용자 INPUT: "이 영화는 지루한가요?"  -> "벡터 DB에서 검색 (지루함과 관련된 노드) " -> "이 영화는 매우 지루했습니다."라는 문장 검색. 
# LLM OUTPUT: "네, 이 영화는 매우 지루하다고 평가되었습니다."

# 결론: 라마인덱스: 데이터 로드, 텍스트 분할, 벡터화, 사용자 질의에 대한 답변 생성까지 쉽고 체계적으로 구성.
